# Обучение детектора ягод на бесплатном GPU (Google Colab)

Обучаем YOLO на своём датасете на видеокарте Colab, скачиваем готовый `best.pt`
и подставляем его локально в `stereo_detect.py`. Видеокарта нужна только здесь;
сам робот работает на CPU.

**Перед началом включите GPU:** меню `Среда выполнения` → `Сменить среду выполнения` → `T4 GPU`.

**Подготовьте датасет.** На своём компьютере в папке `fruit-picker` выполните:
```bash
zip -r dataset.zip dataset -x 'dataset/preview/*'
```
Файл `dataset.zip` загрузите в ячейке ниже.

## 1. Проверяем, что GPU включён

In [ ]:
!nvidia-smi -L
import torch
print('CUDA доступна:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU не включён: Среда выполнения -> Сменить среду выполнения -> T4 GPU'

## 2. Ставим ultralytics (YOLO)

In [ ]:
!pip install -q ultralytics

## 3. Загружаем и распаковываем dataset.zip
После запуска ячейки нажмите кнопку выбора файла и укажите свой `dataset.zip`.

In [ ]:
from google.colab import files
import zipfile, os, shutil

if os.path.exists('/content/dataset'):
    shutil.rmtree('/content/dataset')

uploaded = files.upload()  # выберите dataset.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content')

print('картинок:', len(os.listdir('/content/dataset/images')))
print('файлов разметки:', len(os.listdir('/content/dataset/labels')))

## 4. Чиним пути под Colab
`train.txt`/`val.txt` содержат пути с вашего компьютера. Переписываем их на пути
Colab, сохраняя ту же разбивку train/val, и прописываем `path` в `data.yaml`.

In [ ]:
import yaml
from pathlib import Path

root = Path('/content/dataset')
img_dir = root / 'images'

for split in ['train', 'val']:
    src = root / f'{split}.txt'
    lines = []
    if src.exists():
        for line in src.read_text().splitlines():
            p = img_dir / Path(line).name  # берём только имя файла
            if p.exists():
                lines.append(str(p))
    src.write_text('\n'.join(lines))
    print(split, '->', len(lines), 'фото')

cfg = yaml.safe_load((root / 'data.yaml').read_text())
cfg['path'] = str(root)
(root / 'data.yaml').write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))
print('классы:', cfg['names'])

## 5. Обучение
На T4 это в разы быстрее CPU. `EPOCHS` можно уменьшить для пробы или увеличить
при большом датасете. `batch=16` подходит для T4; если словите нехватку памяти —
снизьте до 8.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
model.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    name='berries',
    patience=20,
    exist_ok=True,
)

## 6. Скачиваем обученную модель
Файл `best.pt` кладём в папку `fruit-picker` рядом со скриптами и подставляем
в `stereo_detect.py` (см. README).

In [ ]:
from google.colab import files
files.download('/content/runs/detect/berries/weights/best.pt')